### In this mini notebook , the purpose is to concatenate and clean various Durban air quality datasets {https://open.africa/dataset/sensorsafrica-airquality-archive-durban} , here's how I went about it:
      1. Created a function to concatenate the datasets and also find the most common sensor ID in the datasets
      2. Created a second function that clean the data and filters it down to just "P2" value types. 

In [237]:
# Import libraries 
import pandas as pd
import pytz
import glob

In [238]:
def concat_files(folder_path, pattern="*.csv", reader=pd.read_csv, cleaner=None):
    files = glob.glob(f"{folder_path}/{pattern}")
    
    # Read *all* files once
    dfs = [reader(f) for f in files]

    # Find global most common sensor ID
    combined = pd.concat(dfs, ignore_index=True)
    global_id = combined["sensor_id"].value_counts().idxmax()

    # Clean each file using the same sensor ID
    if cleaner is not None:
        dfs = [cleaner(df, global_id) for df in dfs]

    return pd.concat(dfs, ignore_index=False)

In [241]:
def clean_df(df,sensor_id):
    # Get data from the sensor with the most entries 
    df = df[df["sensor_id"] == sensor_id]
    
    # Reduce data to only P2 value type
    df = df[df["value_type"] == "P2"]

    # Drop columns not needed 
    df = df.drop(columns=["value_type", "sensor_id", "sensor_type", "location", "lat", "lon"])

    # Set timestamp as index
    df = df.set_index("timestamp")

    # Convert index (timestamp) to a datetime from obj
    df.index = pd.to_datetime(df.index)

    # Localize to UTC if naive
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    
    #Localize time 
    df.index = df.index.tz_convert("Africa/Johannesburg")

    # Resample to 1h mean and foward fill any missing values 
    df = df["value"].resample("1h").mean().ffill().to_frame()

    # Create a lag Feature
    df["value.L1"] = df["value"].shift(1)

    # Drop NaN values
    df.dropna(inplace=True)
    
    return df

In [242]:
df = concat_files("air-quality durban/", reader=lambda f: pd.read_csv(f, sep=';'), cleaner=clean_df)

In [243]:
df.head()

,value,value.L1
timestamp,,
2017-08-18 10:00:00+02:00,290.104720,175.137876
2017-08-18 11:00:00+02:00,279.591680,290.104720
2017-08-18 12:00:00+02:00,292.920521,279.591680
2017-08-18 13:00:00+02:00,306.877927,292.920521
2017-08-18 14:00:00+02:00,295.515475,306.877927


In [244]:
df.tail()

,value,value.L1
timestamp,,
2017-11-30 05:00:00+02:00,8.766667,8.766667
2017-11-30 06:00:00+02:00,8.766667,8.766667
2017-11-30 07:00:00+02:00,8.766667,8.766667
2017-11-30 08:00:00+02:00,11.167500,8.766667
2017-11-30 09:00:00+02:00,7.964118,11.167500


In [245]:
df.shape

(794, 2)

In [246]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 794 entries, 2017-08-18 10:00:00+02:00 to 2017-11-30 09:00:00+02:00
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   value     794 non-null    float64
 1   value.L1  794 non-null    float64
dtypes: float64(2)
memory usage: 18.6 KB
